In [1]:
!pip install -q google-genai pydantic

In [2]:
import pandas as pd
import numpy as np
import json
import time
import getpass

from google import genai
from google.genai import types

from pydantic import BaseModel, Field, ValidationError

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Enter API key securely at runtime

API_KEY = getpass.getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

print("Gemini API client created successfully!")

Paste your Gemini API key: ··········
Gemini API client created successfully!


In [4]:
# Gemini Flash model

MODEL_NAME = "gemini-2.5-flash"

# JSON response schema

SCHEMA = {
    "type": "object",

    "properties": {
        "root_cause": {
            "type": "string"
        },

        "confidence": {
            "type": "number"
        },

        "summary": {
            "type": "string"
        },

        "recommended_action": {
            "type": "string"
        }
    },

    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

print("Model and JSON schema defined!")

Model and JSON schema defined!


In [5]:
SYSTEM_TEXT = """
You are a healthcare appointment follow-up analysis assistant.

Your job is to analyse call-centre notes about missed appointments.

Use only the information available in the note.
Do not invent facts.
Do not make medical diagnoses.
Do not assume a reason when the note is unclear.

Return structured JSON only.
"""

def build_craft_prompt(note_text):

    prompt = f"""
CONTEXT:
A clinic wants to understand why patients missed appointments.

ROLE:
You are a careful healthcare call-centre note analyst.

ACTION:
Read the following patient follow-up note and identify the most likely reason.

Allowed root_cause values:
1. forgot_or_no_reminder
2. distance_or_transport
3. cost_or_fee
4. illness_or_emergency
5. work_or_schedule
6. other

If the reason is unclear or ambiguous, choose "other" rather than guessing.

TONE:
Be neutral, professional and concise.

FORMAT:
Return a JSON object with exactly these four fields:
root_cause, confidence, summary, recommended_action.

Confidence must be between 0 and 1.
Summary must contain at most 25 words.
Do not include extra explanations outside JSON.

PATIENT NOTE:
{note_text}
"""

    return prompt

print("CRAFT prompt created!")

CRAFT prompt created!


In [6]:
# Count API calls

api_call_count = 0

def analyse_note(note_text):

    global api_call_count

    prompt = build_craft_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,

        contents=prompt,

        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA,

            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    api_call_count += 1

    # Parse JSON response
    data = json.loads(response.text)

    return data

In [7]:
# Load notes dataset

notes_df = pd.read_csv("followup_notes.csv")

print("Notes dataset loaded!")
print(notes_df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'followup_notes.csv'

In [8]:
# Get N001 note

n001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

print("N001 Note:")
print(n001)

NameError: name 'notes_df' is not defined

In [9]:
# Analyse N001

n001_result = analyse_note(str(n001))

print("N001 Analysis:")
print(n001_result)

print("\nAPI Calls Used:", api_call_count)

NameError: name 'n001' is not defined

In [10]:
{
    "root_cause": "forgot_or_no_reminder",
    "confidence": 0.95,
    "summary": "...",
    "recommended_action": "..."
}

{'root_cause': 'forgot_or_no_reminder',
 'confidence': 0.95,
 'summary': '...',
 'recommended_action': '...'}

In [11]:
# Allowed root cause categories

ALLOWED_ROOT_CAUSES = [
    "forgot_or_no_reminder",
    "distance_or_transport",
    "cost_or_fee",
    "illness_or_emergency",
    "work_or_schedule",
    "other"
]


# Pydantic validation model

class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)

    confidence: float = Field(ge=0, le=1)

    summary: str = Field(min_length=5)

    recommended_action: str = Field(min_length=5)


def check_root_cause(root_cause):

    if root_cause not in ALLOWED_ROOT_CAUSES:
        raise ValueError(
            f"Invalid root_cause: {root_cause}"
        )

    return True


print("Pydantic model created successfully!")

Pydantic model created successfully!


In [12]:
# Store the exact response text

last_raw_response = ""

def analyse_note(note_text):

    global api_call_count
    global last_raw_response

    prompt = build_craft_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,

        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA,

            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    api_call_count += 1

    # Preserve exact raw JSON text
    last_raw_response = response.text

    # Parse JSON
    data = json.loads(response.text)

    return data

In [13]:
# Process first 10 notes

results = []

for index, row in notes_df.head(10).iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    print("Processing:", note_id)

    # Default values
    data = {}
    raw_response = ""

    # Check blank note BEFORE converting to string
    if pd.isna(note_text) or str(note_text).strip() == "":

        data = {
            "root_cause": "other",
            "confidence": 0.0,
            "summary": "No note recorded",
            "recommended_action": "Re-call patient"
        }

        validation_status = "skipped_blank"

        print("Blank note. API call skipped.")

    else:

        try:

            # Reuse N001 result from Task 4.1
            if note_id == "N001":

                data = n001_result
                raw_response = last_raw_response

            else:

                # Wait before making another API call
                time.sleep(13)

                data = analyse_note(
                    str(note_text).strip()
                )

                raw_response = last_raw_response

            # Pydantic validation
            insight = FollowupInsight(**data)

            # Allowed root cause validation
            check_root_cause(insight.root_cause)

            validation_status = "valid"

        except Exception as e:

            validation_status = "invalid"

            print("Validation/Error:", str(e))

        # Keep fields exactly as returned.
        # Missing fields remain empty.

    result_row = {
        "note_id": note_id,
        "appointment_id": appointment_id,

        "root_cause": data.get("root_cause", ""),
        "confidence": data.get("confidence", ""),
        "summary": data.get("summary", ""),
        "recommended_action": data.get("recommended_action", ""),

        "validation_status": validation_status,

        "raw_response": raw_response
    }

    results.append(result_row)

    print("Status:", validation_status)
    print("------------------------------")

NameError: name 'notes_df' is not defined

In [14]:
# Create final DataFrame

notes_analysed = pd.DataFrame(results)

# Required column order

notes_analysed = notes_analysed[
    [
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response"
    ]
]

# Save CSV
notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

print("notes_analysed.csv saved successfully!")

display(notes_analysed)

KeyError: "None of [Index(['note_id', 'appointment_id', 'root_cause', 'confidence', 'summary',\n       'recommended_action', 'validation_status', 'raw_response'],\n      dtype='object')] are in the [columns]"

In [15]:
print("Total rows:", len(notes_analysed))

print("\nTotal columns:", len(notes_analysed.columns))

print("\nValidation status:")
print(notes_analysed["validation_status"].value_counts())

print("\nN004 status:")
print(
    notes_analysed.loc[
        notes_analysed["note_id"] == "N004",
        "validation_status"
    ]
)

print("\nAPI calls used so far:", api_call_count)

Total rows: 0

Total columns: 0

Validation status:


KeyError: 'validation_status'

In [16]:
from pydantic import ValidationError

# Bad response 1: confidence greater than 1

bad_reply_1 = {
    "root_cause": "other",
    "confidence": 1.7,
    "summary": "Patient did not attend",
    "recommended_action": "Call the patient again"
}

try:

    insight1 = FollowupInsight(**bad_reply_1)

except ValidationError as e:

    print("BAD REPLY 1 ERROR:")
    print(e)

BAD REPLY 1 ERROR:
1 validation error for FollowupInsight
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.7, input_type=float]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal


In [17]:
# Bad response 2: recommended_action is missing

bad_reply_2 = {
    "root_cause": "other",
    "confidence": 0.8,
    "summary": "Patient reason is unclear"
}

try:

    insight2 = FollowupInsight(**bad_reply_2)

except ValidationError as e:

    print("BAD REPLY 2 ERROR:")
    print(e)

BAD REPLY 2 ERROR:
1 validation error for FollowupInsight
recommended_action
  Field required [type=missing, input_value={'root_cause': 'other', '...ient reason is unclear'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.14/v/missing


In [18]:
def simple_v1_prompt(note_text):

    return f"Classify this note:\n{note_text}"

In [19]:
def compare_prompt(prompt):

    global api_call_count

    response = client.models.generate_content(
        model=MODEL_NAME,

        contents=prompt,

        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA,

            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    api_call_count += 1

    return response.text

In [20]:
def compare_prompt(prompt):

    global api_call_count

    response = client.models.generate_content(
        model=MODEL_NAME,

        contents=prompt,

        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA,

            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    api_call_count += 1

    return response.text

In [21]:
# Get N007 and N009 notes

n007 = notes_df.loc[
    notes_df["note_id"] == "N007",
    "note_text"
].iloc[0]

n009 = notes_df.loc[
    notes_df["note_id"] == "N009",
    "note_text"
].iloc[0]

print("N007:")
print(n007)

print("\nN009:")
print(n009)

NameError: name 'notes_df' is not defined

In [22]:
# V1: Simple prompt for N007

v1_n007_raw = compare_prompt(
    simple_v1_prompt(n007)
)

v1_n007 = json.loads(v1_n007_raw)

time.sleep(13)


# V2: CRAFT prompt for N007

v2_n007_raw = compare_prompt(
    build_craft_prompt(n007)
)

v2_n007 = json.loads(v2_n007_raw)

time.sleep(13)


# V1: Simple prompt for N009

v1_n009_raw = compare_prompt(
    simple_v1_prompt(n009)
)

v1_n009 = json.loads(v1_n009_raw)

time.sleep(13)


# V2: CRAFT prompt for N009

v2_n009_raw = compare_prompt(
    build_craft_prompt(n009)
)

v2_n009 = json.loads(v2_n009_raw)

print("Prompt comparison completed!")

NameError: name 'n007' is not defined

In [23]:
# Check whether root cause is allowed

def is_allowed(value):

    return value in ALLOWED_ROOT_CAUSES


comparison_rows = [
    {
        "Note": "N007",
        "Prompt": "V1 - Simple",
        "Root Cause": v1_n007.get("root_cause", ""),
        "Allowed": is_allowed(v1_n007.get("root_cause", ""))
    },

    {
        "Note": "N007",
        "Prompt": "V2 - CRAFT",
        "Root Cause": v2_n007.get("root_cause", ""),
        "Allowed": is_allowed(v2_n007.get("root_cause", ""))
    },

    {
        "Note": "N009",
        "Prompt": "V1 - Simple",
        "Root Cause": v1_n009.get("root_cause", ""),
        "Allowed": is_allowed(v1_n009.get("root_cause", ""))
    },

    {
        "Note": "N009",
        "Prompt": "V2 - CRAFT",
        "Root Cause": v2_n009.get("root_cause", ""),
        "Allowed": is_allowed(v2_n009.get("root_cause", ""))
    }
]

prompt_comparison = pd.DataFrame(comparison_rows)

display(prompt_comparison)

NameError: name 'v1_n007' is not defined

In [24]:
print("TOTAL GEMINI API CALLS")
print("----------------------")
print("API calls used:", api_call_count)

TOTAL GEMINI API CALLS
----------------------
API calls used: 0


In [25]:
except Exception as e:

    validation_status = "invalid"
    raw_response = last_raw_response

    print("Validation/Error:", str(e))

SyntaxError: invalid syntax (3154740718.py, line 1)

In [26]:
# Load cleaned appointment data

appointments_df = pd.read_csv(
    "appointments_clean.csv"
)

# Load GenAI analysed notes

notes_analysed = pd.read_csv(
    "notes_analysed.csv"
)

print("Both datasets loaded successfully!")

print("Appointments:", appointments_df.shape)
print("Analysed notes:", notes_analysed.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'appointments_clean.csv'

In [27]:
# Merge GenAI results with appointment data

merged_df = notes_analysed.merge(
    appointments_df,
    on="appointment_id",
    how="left",
    suffixes=("_note", "_appointment")
)

print("Datasets merged successfully!")

print("Merged rows:", len(merged_df))

display(
    merged_df[
        [
            "note_id",
            "appointment_id",
            "root_cause",
            "reminder_sent"
        ]
    ]
)

NameError: name 'appointments_df' is not defined

In [28]:
# Cross-tabulation of root cause and reminder status

root_cause_reminder_table = pd.crosstab(
    merged_df["root_cause"],
    merged_df["reminder_sent"]
)

print("ROOT CAUSE VS REMINDER SENT")

display(root_cause_reminder_table)

NameError: name 'merged_df' is not defined

In [29]:
# Check forgotten/no-reminder appointments

forgotten_notes = merged_df[
    merged_df["root_cause"] == "forgot_or_no_reminder"
]

print("Forgot/no-reminder records:")

display(
    forgotten_notes[
        [
            "note_id",
            "appointment_id",
            "root_cause",
            "reminder_sent"
        ]
    ]
)

print(
    "\nTotal appointments with no reminder:",
    (merged_df["reminder_sent"] == "No").sum()
)

NameError: name 'merged_df' is not defined